# Predicting Student Test Scores 
## Score: 8.71257

In [1]:
import time
import numpy as np
import pandas as pd

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LogisticRegression


In [2]:
train = pd.read_csv('playground-series-s6e1/train.csv')
test = pd.read_csv('playground-series-s6e1/test.csv')

test_ids = test['id'].to_numpy()

y = train['exam_score'].to_numpy(dtype=float)

X = train.drop(columns=['id', 'exam_score'])
X_test = test.drop(columns=['id'])

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')


In [3]:
base_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'n_estimators': 5000,
    'num_leaves': 79,
    'max_depth': 10,
    'min_child_samples': 55,
    'reg_alpha': 10.0,
    'reg_lambda': 0.50,
    'min_split_gain': 1e-6,
    'subsample': 0.72,
    'subsample_freq': 3,
    'colsample_bytree': 0.65,
    'n_jobs': -1,
    'force_col_wise': True
}

seeds = [420, 666]
n_splits = 5

EARLY_STOP = 175
MAX_SECONDS = 3300

TE_SMOOTH = 25.0

te_cols = [c for c in ['course', 'exam_difficulty', 'study_method', 'sleep_quality', 'facility_rating', 'internet_access', 'gender'] if c in X.columns]
te_pairs = []
for a, b in [('course', 'exam_difficulty'), ('study_method', 'exam_difficulty'), ('course', 'study_method')]:
    if a in X.columns and b in X.columns:
        te_pairs.append((a, b))

t0 = time.time()

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

def te_fit_1(X_ref, y_ref, col, smooth):
    y_s = pd.Series(y_ref, index=X_ref.index)
    g = y_s.groupby(X_ref[col]).agg(['mean', 'count'])
    prior = float(y_s.mean())
    enc = (g['mean'] * g['count'] + prior * smooth) / (g['count'] + smooth)
    return enc, prior

def te_fit_2(X_ref, y_ref, a, b, smooth):
    y_s = pd.Series(y_ref, index=X_ref.index)
    key = X_ref[a].astype(str) + '|' + X_ref[b].astype(str)
    g = y_s.groupby(key).agg(['mean', 'count'])
    prior = float(y_s.mean())
    enc = (g['mean'] * g['count'] + prior * smooth) / (g['count'] + smooth)
    return enc, prior

def te_apply_1(X_df, col, enc, prior):
    return X_df[col].map(enc).astype(float).fillna(prior)

def te_apply_2(X_df, a, b, enc, prior):
    key = X_df[a].astype(str) + '|' + X_df[b].astype(str)
    return key.map(enc).astype(float).fillna(prior)

def add_te(X_tr, X_va, X_te, y_tr):
    add_tr = {}
    add_va = {}
    add_te2 = {}

    for c in te_cols:
        enc, prior = te_fit_1(X_tr, y_tr, c, TE_SMOOTH)
        name = f'te_{c}'
        add_tr[name] = te_apply_1(X_tr, c, enc, prior)
        add_va[name] = te_apply_1(X_va, c, enc, prior)
        add_te2[name] = te_apply_1(X_te, c, enc, prior)

    for a, b in te_pairs:
        enc, prior = te_fit_2(X_tr, y_tr, a, b, TE_SMOOTH)
        name = f'te_{a}__{b}'
        add_tr[name] = te_apply_2(X_tr, a, b, enc, prior)
        add_va[name] = te_apply_2(X_va, a, b, enc, prior)
        add_te2[name] = te_apply_2(X_te, a, b, enc, prior)

    X_tr2 = pd.concat([X_tr, pd.DataFrame(add_tr, index=X_tr.index)], axis=1)
    X_va2 = pd.concat([X_va, pd.DataFrame(add_va, index=X_va.index)], axis=1)
    X_te2 = pd.concat([X_te, pd.DataFrame(add_te2, index=X_te.index)], axis=1)

    return X_tr2, X_va2, X_te2

def cv_run(params, seeds, label):
    sum_oof = np.zeros(len(X), dtype=float)
    cnt_oof = np.zeros(len(X), dtype=float)
    sum_test = np.zeros(len(X_test), dtype=float)
    seeds_done = 0

    for s_i, seed in enumerate(seeds, start=1):
        p = {**params, 'random_state': seed}

        oof = np.full(len(X), np.nan, dtype=float)
        test_pred_sum = np.zeros(len(X_test), dtype=float)
        rmse_scores = []
        folds_done = 0

        print(f'{label} SEED {seed} ({s_i}/{len(seeds)})')

        for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
            if (time.time() - t0) > MAX_SECONDS:
                break

            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            X_tr2, X_va2, X_test2 = add_te(X_tr, X_va, X_test, y_tr)

            model = lgb.LGBMRegressor(**p)
            model.fit(
                X_tr2,
                y_tr,
                eval_set=[(X_va2, y_va)],
                callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(0)]
            )

            va_pred = model.predict(X_va2)
            oof[va_idx] = va_pred

            fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
            rmse_scores.append(fold_rmse)
            print(f'  Fold {fold}/{n_splits} RMSE: {fold_rmse:.5f}')

            test_pred_sum += model.predict(X_test2)
            folds_done += 1

        if folds_done == 0:
            break

        test_pred = test_pred_sum / folds_done

        filled = ~np.isnan(oof)
        oof_filled = np.clip(oof[filled], 0, 100)
        sum_oof[filled] += oof_filled
        cnt_oof[filled] += 1.0

        sum_test += np.clip(test_pred, 0, 100)
        seeds_done += 1

        oof_rmse = float(np.sqrt(mean_squared_error(y[filled], oof_filled)))
        print(f'{label} Seed {seed} OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})')

        if (time.time() - t0) > MAX_SECONDS:
            break

    denom = np.maximum(cnt_oof, 1.0)
    all_oof = np.clip(sum_oof / denom, 0, 100)

    if seeds_done > 0:
        all_test = np.clip(sum_test / seeds_done, 0, 100)
    else:
        all_test = np.zeros(len(X_test), dtype=float)

    filled_all = cnt_oof > 0
    if filled_all.any():
        final_oof_rmse = float(np.sqrt(mean_squared_error(y[filled_all], all_oof[filled_all])))
    else:
        final_oof_rmse = float('nan')

    print(f'{label} FINAL OOF RMSE: {final_oof_rmse:.5f}')

    return all_oof, all_test, final_oof_rmse


oof, test_pred, _ = cv_run(base_params, seeds, 'LGB')

oof_rmse = float(np.sqrt(mean_squared_error(y, oof)))
print(f'OOF RMSE: {oof_rmse:.5f}')

submission = pd.DataFrame({'id': test_ids, 'exam_score': np.clip(test_pred, 0, 100)})
submission.to_csv('submission.csv', index=False)
print('Wrote submission.csv')


LGB SEED 420 (1/2)


C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = y_s.groupby(X_ref[col]).agg(['mean', 'count'])
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = y_s.groupby(X_ref[col]).agg(['mean', 'count'])
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the

Training until validation scores don't improve for 175 rounds
Early stopping, best iteration is:
[1710]	valid_0's rmse: 8.75148
  Fold 1/5 RMSE: 8.75148


C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = y_s.groupby(X_ref[col]).agg(['mean', 'count'])
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = y_s.groupby(X_ref[col]).agg(['mean', 'count'])
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the

Training until validation scores don't improve for 175 rounds
Early stopping, best iteration is:
[1708]	valid_0's rmse: 8.7501
  Fold 2/5 RMSE: 8.75010


...

Training until validation scores don't improve for 175 rounds
Early stopping, best iteration is:
[1451]	valid_0's rmse: 8.74745
  Fold 3/5 RMSE: 8.74745


C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = y_s.groupby(X_ref[col]).agg(['mean', 'count'])
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = y_s.groupby(X_ref[col]).agg(['mean', 'count'])
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the

Training until validation scores don't improve for 175 rounds
Early stopping, best iteration is:
[1264]	valid_0's rmse: 8.75905
  Fold 4/5 RMSE: 8.75905


C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = y_s.groupby(X_ref[col]).agg(['mean', 'count'])
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = y_s.groupby(X_ref[col]).agg(['mean', 'count'])
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_41724\4006317399.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the

Training until validation scores don't improve for 175 rounds
Early stopping, best iteration is:
[1373]	valid_0's rmse: 8.77952
  Fold 5/5 RMSE: 8.77952
LGB Seed 666 OOF RMSE: 8.75771 | Mean fold: 8.75791 (+/- 0.01143)
LGB FINAL OOF RMSE: 8.75290
OOF RMSE: 8.75290
Wrote submission.csv
